# Can It Notice Its Own Thoughts? Qwen2.5 family on a free Kaggle GPU

Settings: Accelerator = GPU T4, Internet = on (needs a phone-verified account). The first cell pulls the trials Colab finished before its GPU quota ran out (branch `partial-run`), so the sweep resumes instead of starting over. Use **Save Version > Save & Run All** so it runs without the browser open; results land in Output.

In [ ]:
import os, zipfile, urllib.request
RUN = '/kaggle/working/introspect-runs'
os.makedirs(RUN, exist_ok=True)
URL = 'https://raw.githubusercontent.com/feliciachen01/introspect/partial-run/qwen-family-partial.zip'
if not os.path.exists(f'{RUN}/qwen-family'):
    urllib.request.urlretrieve(URL, '/kaggle/working/partial.zip')
    zipfile.ZipFile('/kaggle/working/partial.zip').extractall(RUN)
    os.remove('/kaggle/working/partial.zip')
!find {RUN} -name trials.jsonl | xargs -r wc -l

In [ ]:
!git clone -q https://github.com/feliciachen01/introspect.git /kaggle/working/introspect
%cd /kaggle/working/introspect
!pip install -q -e . bitsandbytes
!pytest -q

In [ ]:
# 1. Generate: concept vectors + injected / control / bias trials (resumes). One GPU: the 7B fits in 4-bit.
!CUDA_VISIBLE_DEVICES=0 introspect sweep --config configs/kaggle.yaml

In [ ]:
# 2. Grade with vLLM. Its install can leave torchaudio built for another CUDA; transformers imports it
# if present, so remove it. If vLLM still fails, the grader falls back to transformers (slower).
!pip install -q vllm
!pip uninstall -y -q torchaudio
!introspect judge --config configs/kaggle.yaml

In [ ]:
# 3. Report
!introspect report --config configs/kaggle.yaml
from IPython.display import Markdown, display
display(Markdown(open('/kaggle/working/introspect-runs/qwen-family/results.md').read()))

In [ ]:
# keep Output small: drop the cloned repo and the large concept-vector files
!rm -rf /kaggle/working/introspect
!find /kaggle/working/introspect-runs -name '*.npz' -delete
!du -sh /kaggle/working/introspect-runs